In [1]:
# !pip install --no-cache-dir --no-deps geopandas
# !pip install --no-cache-dir --no-deps plotly
# !pip install pandas numpy matplotlib shapely fiona pyproj rtree
# !pip install ipywidgets
#!pip install pandasql

In [2]:
#%% Imports
import pandas as pd
from pandasql import sqldf
import numpy as np 
import geopandas as gpd
import matplotlib.pyplot as plt
import plotly as px

In [3]:
def sql_query(q):
    return sqldf(q, globals())

In [4]:
# %%
crop_yield = pd.read_csv('US Crop Yield (in $) by County, 1997-2022.csv')
crop_yield = crop_yield[['Year', 'State', 'State ANSI', 'County', 'County ANSI', 'Value']]


In [5]:
years = np.arange(1970, 2025)
zindex = {}
for year in years:
    try:
        zindex[year] = pd.read_csv(f'zIndex/{year} z index data.csv', 
                                comment='#',  
                                skiprows=3)  
        zindex[year] = zindex[year][['ID', 'Value', 'Anomaly (1901-2000 base period)']]
    except Exception as e:
        print(f"Error reading file for year {year}: {e}")

Error reading file for year 1977: [Errno 2] No such file or directory: 'zIndex/1977 z index data.csv'


In [6]:
zindex[2000].head()

,ID,Value,Anomaly (1901-2000 base period)
0,AL-001,-0.20,-0.16
1,AL-003,-0.96,-0.97
2,AL-005,-0.45,-0.46
3,AL-007,-0.70,-0.66
4,AL-009,-0.67,-0.71


In [8]:
column_mapping = {
    'ID': 'County ANSI',
    'Value': 'ZIndexValue',
    'Anomaly (1901-2000 base period)': 'Anomaly'
}

for year in zindex:
    zindex[year] = zindex[year].rename(columns=column_mapping)
    zindex[year]['County ANSI'] = zindex[year]['County ANSI'].str.split('-').str[1].astype(int)

AttributeError: Can only use .str accessor with string values!

In [9]:
crop_yield.head()

,Year,State,State ANSI,County,County ANSI,Value
0,2022,NaN,NaN,NaN,NaN,(D)
1,2022,NaN,NaN,NaN,NaN,"231,000"
2,2022,NaN,NaN,NaN,NaN,"52,569,000"
3,2022,NaN,NaN,NaN,NaN,(D)
4,2022,NaN,NaN,NaN,NaN,(D)


In [10]:
crop_yield.dropna(subset=['County ANSI'], inplace=True)
crop_yield = crop_yield[~crop_yield['Value'].str.contains(r'\(D\)', na=False)]

In [11]:
crop_yield.head(100)

,Year,State,State ANSI,County,County ANSI,Value
68,2022,ALABAMA,1.0,AUTAUGA,1.0,"29,212,000"
69,2022,ALABAMA,1.0,BULLOCK,11.0,"36,100,000"
70,2022,ALABAMA,1.0,DALLAS,47.0,"40,553,000"
71,2022,ALABAMA,1.0,ELMORE,51.0,"24,592,000"
72,2022,ALABAMA,1.0,GREENE,63.0,"2,405,000"
...,...,...,...,...,...,...
171,2022,ARKANSAS,5.0,SAINT FRANCIS,123.0,"203,112,000"
173,2022,ARKANSAS,5.0,BAXTER,5.0,"1,317,000"
174,2022,ARKANSAS,5.0,CLEBURNE,23.0,"1,884,000"
175,2022,ARKANSAS,5.0,FULTON,49.0,"1,414,000"


In [12]:
z_index_combined = pd.concat(
    [df.assign(Year=year) for year, df in zindex.items()],
    ignore_index=True
)


In [13]:
z_index_combined = z_index_combined.rename(
    columns={'County ANSI':'County_ANSI'}
)

In [14]:
if 'Value' in z_index_combined.columns:
    z_index_combined = z_index_combined.rename(columns={'Value':'ZIndexValue'})

In [25]:
z_index_unique = (
    z_index_combined
      [['Year','County_ANSI','ZIndexValue', 'Anomaly']]
      .drop_duplicates(subset=['Year','County_ANSI'])
)

In [26]:
crop_yield = crop_yield.rename(
    columns={
      'County ANSI':'County_ANSI',
      'State ANSI' :'State_ANSI',
      'Value'      :'Crop_Yield'
    }
)
crop_yield['County_ANSI'] = crop_yield['County_ANSI'].astype(int)

In [27]:
merged_data = crop_yield.merge(
    z_index_unique,
    on=['Year','County_ANSI'],
    how='left'
)

In [28]:
merged_data.head()


,Year,State,State_ANSI,County,County_ANSI,Crop_Yield,ZIndexValue,Anomaly
0,2022,ALABAMA,1.0,AUTAUGA,1,"29,212,000",0.75,0.79
1,2022,ALABAMA,1.0,BULLOCK,11,"36,100,000",0.77,0.76
2,2022,ALABAMA,1.0,DALLAS,47,"40,553,000",0.39,0.41
3,2022,ALABAMA,1.0,ELMORE,51,"24,592,000",0.95,1.01
4,2022,ALABAMA,1.0,GREENE,63,"2,405,000",1.35,1.37
